# DeBERTa NLI Direction Scoring

Phase A notebook: dùng DeBERTa-v3-large-MNLI zero-shot để chấm chiều prerequisite A→B vs B→A. Input nên là `edge_scoring_input_bundle.json` vì có đủ P5 edges, KP descriptions, và GPT-5.4 labels để eval agreement.


In [ ]:
DRIVE_URL = ""  # Optional: paste a Google Drive link to edge_scoring_input_bundle.json or a zip containing it.
MODEL_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
MAX_LENGTH = 512
BATCH_SIZE = 8
OUTPUT_PATH = "/content/deberta_nli_edge_direction_scores.json"


In [ ]:
!pip -q install -U "transformers>=4.57.0" accelerate gdown safetensors sentencepiece protobuf


In [ ]:
import json
import re
import shutil
import zipfile
from pathlib import Path

import requests

def extract_drive_file_id(url: str) -> str:
    patterns = [r"/file/d/([A-Za-z0-9_-]+)", r"[?&]id=([A-Za-z0-9_-]+)"]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Cannot extract Google Drive file id from: {url}")

def download_from_drive(file_id: str, output_path: Path) -> None:
    import gdown
    result = gdown.download(id=file_id, output=str(output_path), quiet=False)
    if result and output_path.exists() and output_path.stat().st_size > 0:
        return
    session = requests.Session()
    url = "https://drive.google.com/uc?export=download"
    response = session.get(url, params={"id": file_id}, stream=True)
    token = None
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            token = value
            break
    if token:
        response = session.get(url, params={"id": file_id, "confirm": token}, stream=True)
    response.raise_for_status()
    with output_path.open("wb") as handle:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                handle.write(chunk)

def load_payload() -> dict:
    local_candidates = [
        Path("/content/edge_scoring_input_bundle.json"),
        Path("edge_scoring_input_bundle.json"),
        Path("/content/p5_drive_input"),
    ]
    for path in local_candidates:
        if path.exists() and path.stat().st_size > 0:
            print("Loading local input:", path)
            return json.loads(path.read_text(encoding="utf-8"))
    if not DRIVE_URL.strip():
        raise FileNotFoundError("Upload edge_scoring_input_bundle.json to /content or set DRIVE_URL.")
    download_path = Path("/content/p5_drive_input")
    extract_dir = Path("/content/p5_drive_extract")
    if download_path.exists(): download_path.unlink()
    if extract_dir.exists(): shutil.rmtree(extract_dir)
    file_id = extract_drive_file_id(DRIVE_URL)
    print("Drive file id:", file_id)
    download_from_drive(file_id, download_path)
    if zipfile.is_zipfile(download_path):
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(download_path) as zf:
            zf.extractall(extract_dir)
        json_files = sorted(extract_dir.rglob("*.json"))
        if not json_files:
            raise FileNotFoundError("Zip has no JSON files")
        input_path = json_files[0]
    else:
        input_path = download_path
    print("Loading downloaded input:", input_path)
    return json.loads(input_path.read_text(encoding="utf-8"))

payload = load_payload()
print("Top-level keys:", list(payload.keys()))
print("Input stats:", payload.get("input_stats"))


In [ ]:
clean_edges = payload.get("clean_candidate_edges")
if not isinstance(clean_edges, list):
    raise ValueError("Expected clean_candidate_edges[]")

global_kps = payload.get("global_kps") or payload.get("concepts_kp_global") or []
kp_index = {kp.get("global_kp_id") or kp.get("kp_id"): kp for kp in global_kps if isinstance(kp, dict) and (kp.get("global_kp_id") or kp.get("kp_id"))}
gpt_index = {(r["source_kp_id"], r["target_kp_id"]): r for r in payload.get("gpt54_edge_labels", []) if isinstance(r, dict) and r.get("source_kp_id") and r.get("target_kp_id")}

def readable_kp_id(kp_id: str) -> str:
    return re.sub(r"[_-]+", " ", kp_id.removeprefix("kp_")).strip()

def kp_text(kp_id: str) -> str:
    kp = kp_index.get(kp_id, {})
    name = kp.get("name") or kp.get("canonical_name") or readable_kp_id(kp_id)
    desc = kp.get("description") or kp.get("global_description") or ""
    return f"{name}. {desc}".strip()

def nli_pair(edge: dict, *, reverse: bool = False):
    source = edge["target_kp_id"] if reverse else edge["source_kp_id"]
    target = edge["source_kp_id"] if reverse else edge["target_kp_id"]
    premise = f"Source concept: {kp_text(source)} Target concept: {kp_text(target)}"
    hypothesis = f"Understanding {kp_text(source)} is required before understanding {kp_text(target)}."
    return premise, hypothesis

print("Clean edges:", len(clean_edges))
print("KP catalog entries:", len(kp_index))
print("GPT labels:", len(gpt_index))
print("Sample pair:", nli_pair(clean_edges[0])[0][:500], "\nH:", nli_pair(clean_edges[0])[1][:300])


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("device=", device, "dtype=", dtype)
if device == "cuda": print("gpu=", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
model.eval()
print("id2label:", model.config.id2label)

label_to_id = {str(v).lower(): int(k) for k, v in model.config.id2label.items()}
entailment_id = next((i for lab, i in label_to_id.items() if "entail" in lab), None)
contradiction_id = next((i for lab, i in label_to_id.items() if "contrad" in lab), None)
neutral_id = next((i for lab, i in label_to_id.items() if "neutral" in lab), None)
if entailment_id is None or contradiction_id is None:
    raise RuntimeError(f"Cannot find entailment/contradiction labels in {model.config.id2label}")
print("entailment_id", entailment_id, "contradiction_id", contradiction_id, "neutral_id", neutral_id)

def score_pairs(pairs, batch_size=BATCH_SIZE):
    out = []
    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]
        premises = [p for p, h in batch]
        hypotheses = [h for p, h in batch]
        encoded = tokenizer(premises, hypotheses, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**encoded).logits.float()
        probs = F.softmax(logits, dim=-1)
        for row in range(probs.shape[0]):
            entail = probs[row, entailment_id].item()
            contra = probs[row, contradiction_id].item()
            neutral = probs[row, neutral_id].item() if neutral_id is not None else 0.0
            directional_score = entail - contra
            strength_score = entail / max(1e-9, entail + contra)
            out.append({"entailment": entail, "contradiction": contra, "neutral": neutral, "directional_score": directional_score, "strength_score": strength_score})
    return out

print("Model loaded")


In [ ]:
forward_pairs = [nli_pair(edge, reverse=False) for edge in clean_edges]
reverse_pairs = [nli_pair(edge, reverse=True) for edge in clean_edges]
forward_scores = score_pairs(forward_pairs)
reverse_scores = score_pairs(reverse_pairs)

scored_edges = []
for edge, fs, rs in zip(clean_edges, forward_scores, reverse_scores):
    direction_margin = fs["directional_score"] - rs["directional_score"]
    nli_direction = "forward" if direction_margin >= 0.15 else ("reverse" if direction_margin <= -0.15 else "ambiguous")
    gpt = gpt_index.get((edge["source_kp_id"], edge["target_kp_id"]), {})
    scored = dict(edge)
    scored.update({
        "nli_model": MODEL_ID,
        "nli_scoring_method": "deberta_mnli_forward_reverse_v1",
        "nli_forward_entailment": round(fs["entailment"], 4),
        "nli_forward_contradiction": round(fs["contradiction"], 4),
        "nli_forward_neutral": round(fs["neutral"], 4),
        "nli_reverse_entailment": round(rs["entailment"], 4),
        "nli_reverse_contradiction": round(rs["contradiction"], 4),
        "nli_reverse_neutral": round(rs["neutral"], 4),
        "nli_edge_strength": round(fs["strength_score"], 4),
        "nli_reverse_strength": round(rs["strength_score"], 4),
        "nli_direction_margin": round(direction_margin, 4),
        "nli_direction_verdict": nli_direction,
        "gpt54_verdict": gpt.get("gpt54_verdict"),
        "gpt54_confidence": gpt.get("gpt54_confidence"),
    })
    scored_edges.append(scored)

print("Scored edges:", len(scored_edges))
print(json.dumps(scored_edges[0], indent=2)[:1800])


In [ ]:
from collections import Counter

margins = [e["nli_direction_margin"] for e in scored_edges]
strengths = [e["nli_edge_strength"] for e in scored_edges]
gpt_available = [e for e in scored_edges if e.get("gpt54_verdict")]
direction_agree_keep = [e for e in gpt_available if e["gpt54_verdict"] == "keep" and e["nli_direction_verdict"] in {"forward", "ambiguous"}]
output = {
    "run_id": payload.get("run_id", "p5_cs224n_cs231n"),
    "stage_id": "deberta_nli_direction_scoring",
    "model_id": MODEL_ID,
    "max_length": MAX_LENGTH,
    "scoring_method": "deberta_mnli_forward_reverse_v1",
    "scored_edges": scored_edges,
    "score_summary": {
        "edge_count": len(scored_edges),
        "nli_direction_verdict_distribution": dict(Counter(e["nli_direction_verdict"] for e in scored_edges)),
        "nli_direction_margin_min": round(min(margins), 4),
        "nli_direction_margin_max": round(max(margins), 4),
        "nli_direction_margin_mean": round(sum(margins) / len(margins), 4),
        "nli_edge_strength_min": round(min(strengths), 4),
        "nli_edge_strength_max": round(max(strengths), 4),
        "nli_edge_strength_mean": round(sum(strengths) / len(strengths), 4),
        "gpt54_label_count": len(gpt_available),
        "gpt54_keep_not_reversed_count": len(direction_agree_keep),
    },
}
Path(OUTPUT_PATH).write_text(json.dumps(output, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(output["score_summary"], indent=2))
print("Wrote", OUTPUT_PATH)


In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
except Exception as exc:
    print("files.download unavailable:", exc)
    print("Output path:", OUTPUT_PATH)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_output = Path("/content/drive/MyDrive/deberta_nli_edge_direction_scores.json")
    shutil.copyfile(OUTPUT_PATH, drive_output)
    print("Saved to Google Drive:", drive_output)
except Exception as exc:
    print("Drive save skipped:", exc)
